# SYNUR: direct JEV observation extraction

Use **local** SYNUR files and JEV's native `state + questions + model` interface. **No separate system prompt is required.** Context and schema are state; each question carries extraction rules, its concept definition, and criteria. Python assembles the typed answers into SYNUR observations.

JEV is the only model. This run enables **SINGLE_SELECT, MULTI_SELECT, and NUMERIC**: only STRING concepts are excluded from prediction, reference labels, and evaluation. The full 193-concept source schema and dataset files stay unchanged. Uncertain/conflicting values go to review. This is not the extractor/verifier cascade.

**Prerequisite:** complete the separate dataset setup in the README. This notebook never downloads data. Credentials are entered through the masked setup prompt or inherited from the environment, never saved in cell source. `LIVE_CALLS` controls model execution. Research only: SYNUR is synthetic nurse dictation, not evidence of accuracy on doctor-patient dialogue.

In [ ]:
import os
from collections import Counter
from pathlib import Path
from uuid import uuid4

from IPython.display import JSON, display
from synur.dataset import load_dataset
from synur.evaluation import evaluate
from synur.experiment import Settings, extract, preview, save_run
from synur.jev import JevAdapter
from synur.observations import VALUE_TYPES, SchemaRegistry, normalize_references
from synur.reporting import save_transcript_report

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Launch Jupyter from the project root or notebooks directory.')
DATA_DIR = Path(os.environ.get('SYNUR_DATA_DIR', str(ROOT / 'data' / 'synur')))
MODEL = os.environ.get('TYPESAFE_MODEL', 'jev-1.13.0')
SPLIT = 'mediqa_synur_dev'
ROW_ID = '152'  # Exact split-local ID, not a positional index; None uses SAMPLE_LIMIT.
SAMPLE_LIMIT = 1
ENABLED_VALUE_TYPES = ('SINGLE_SELECT', 'MULTI_SELECT', 'NUMERIC')
LIVE_CALLS = True  # Explicit opt-in; run the API key setup cell before inference.
SAVE_RESULTS = False
SAVE_REPORT = True
SETTINGS = Settings()
print({'data_dir': str(DATA_DIR), 'model': MODEL, 'live_calls': LIVE_CALLS})

## Local data and schema
Read and verify already-downloaded files. A missing/corrupt file is an explicit error, never an automatic network fetch. `original` overlaps train/dev and must not be treated as extra independent data. Test labels are reserved for a held-out run.

In [ ]:
dataset = load_dataset(DATA_DIR)
full_registry = SchemaRegistry.from_entries(dataset.schema_entries)
if not ENABLED_VALUE_TYPES or any(kind not in VALUE_TYPES for kind in ENABLED_VALUE_TYPES):
    raise ValueError('ENABLED_VALUE_TYPES must contain known observation types.')
excluded_value_types = tuple(kind for kind in VALUE_TYPES if kind not in ENABLED_VALUE_TYPES)
registry = SchemaRegistry(tuple(concept for concept in full_registry.concepts
                                if concept.value_type in ENABLED_VALUE_TYPES))
scoped_splits = {
    split: [{**row, 'observations': [label for label in row['observations']
                                   if label.get('value_type') not in excluded_value_types]}
            for row in split_rows]
    for split, split_rows in dataset.splits.items()
}
if not isinstance(SAMPLE_LIMIT, int) or isinstance(SAMPLE_LIMIT, bool) or SAMPLE_LIMIT < 1:
    raise ValueError('SAMPLE_LIMIT must be a positive integer.')
if ROW_ID is None:
    rows = scoped_splits[SPLIT][:SAMPLE_LIMIT]
else:
    if not isinstance(ROW_ID, str) or not ROW_ID:
        raise ValueError('ROW_ID must be a nonempty string or None.')
    rows = [row for row in scoped_splits[SPLIT] if row['id'] == ROW_ID]
    if len(rows) != 1:
        raise ValueError(f'Expected exactly one row with ID {ROW_ID!r} in {SPLIT}.')
display(JSON({
    'splits': {name: len(values) for name, values in dataset.splits.items()},
    'source_concepts': len(full_registry.concepts),
    'enabled_concepts': len(registry.concepts),
    'excluded_value_types': list(excluded_value_types),
    'excluded_labels': {split: sum(len(raw['observations']) - len(scoped['observations'])
                                   for raw, scoped in zip(dataset.splits[split], split_rows))
                        for split, split_rows in scoped_splits.items()},
    'value_types': dict(Counter(item.value_type for item in registry.concepts)),
    'selected_split': SPLIT,
    'selected_row_ids': [row['id'] for row in rows],
}))

## 1. Transcript and reference labels
Show the selected row's original transcript and categorical/numeric labels before inference. Only STRING observations are ignored, including in the labels below. Raw files stay unchanged. Any conservative label normalization is shown explicitly; both raw and normalized scores are reported later. Labels never enter the model request.

In [ ]:
for row in rows:
    print(f"Transcript: {SPLIT}, row {row['id']}")
    print(row['transcript'])
    print('Reference labels (enabled types only):')
    display(JSON(row['observations']))
    reference_view = normalize_references(row['observations'], registry,
                                          split=SPLIT, row_id=row['id'])
    if reference_view.changes or reference_view.issues:
        display(JSON({'normalized_labels': reference_view.observations,
                      'normalizations': reference_view.changes,
                      'label_issues': reference_view.issues}))
sample = rows[0]

## Native JEV request preview
`state = {'context': transcript, 'schema': registry}`. Questions use `Choice` or `Noul`, with reusable extraction rules in `instructions` and alternatives in `criteria`. Question IDs only correlate answers; the concept definition is always in the instructions.

Stage A classifies support/absence/ambiguity/conflict for every enabled concept. Stage B resolves supported enum values and selects numeric scalars from the transcript. Disabled STRING concepts are absent from the model schema and questions; text-span candidates are not generated. Output confidence and probability thresholds are experimental, not clinically calibrated.

In [ ]:
request_preview = preview(sample['transcript'], registry, SETTINGS)
display(JSON({'model': MODEL,
              'state': request_preview['state'],
              'concept_count': request_preview['concept_count'],
              'status_batches': request_preview['status_batches'],
              'first_status_questions': request_preview['questions'][:2],
              'value_question_examples': request_preview['value_question_examples'],
              'numeric_candidates': request_preview['numeric_candidates'],
              'candidate_issues': request_preview['candidate_issues'],
              'settings': request_preview['settings']}))
assert request_preview['concept_count'] == len(registry.concepts)

## 2. Model output observations (live JEV)
Set `LIVE_CALLS = True` above, then run the API key setup cell below. It reuses a valid-looking `TYPESAFE_API_KEY` from the kernel environment or asks for the key in a masked input box. Paste the key into that box, not into cell source. No key is written to the notebook or a file. The key lasts only for this kernel session; restart the kernel to clear a key entered here. Configuration does not verify the key with the service or make model calls. Never send real patient text without authorization.

The emitted observation contract is `[{id, name, value_type, value}, ...]`. Review status, probabilities, source offsets, and failures stay separate. A failed request is not an empty successful extraction. Raw SDK responses are validated before interpretation.

In [ ]:
from synur.jev import configure_api_key

configure_api_key(enabled=LIVE_CALLS)
if LIVE_CALLS:
    print('API key configured for this kernel session; its value is not displayed.')
else:
    print('Live calls disabled; no API key requested.')

In [ ]:
predictions = []
if LIVE_CALLS:
    with JevAdapter(enabled=True, model=MODEL) as jev:
        predictions = [extract(row['transcript'], registry, jev, row_id=row['id'],
                               settings=SETTINGS) for row in rows]
    for result in predictions:
        print(f"Model observations: row {result['id']} ({result['status']})")
        display(JSON(result['observations']))
        if result['failures']:
            display(JSON({'failures': result['failures']}))
else:
    print('NOT RUN: live JEV calls are disabled. No predictions or model accuracy are claimed.')

## 3. Error counts, precision, recall, and F1
Local baseline scores, **not the official shared-task scorer**. Categorical and NUMERIC observations are scored; only STRING labels are excluded from false negatives and metric denominators. Raw and normalized reference scores use the same filtered labels; multi-select order is ignored. Numeric values compare exactly (150 equals 150.0), without tolerance or unit conversion. Abstentions and failed/missing rows still affect recall for enabled types. Review rate and coverage use only enabled concepts. Numeric candidate coverage is a separate source-discovery diagnostic, not model accuracy.

**Correct (C):** exact concept and value match. **Substitution (S):** the same concept ID with a different value or invalid observation. **Insertion (I):** an extra predicted observation after substitutions are paired. **Deletion (D):** an unmatched reference observation. Each multi-select list is one observation, not one score per member. Matching is within each row, never across rows.

`precision = C / (C + I + S)`, `recall = C / (C + D + S)`, `F1 = 2C / (2C + I + D + 2S)`. A substitution contributes one false positive and one false negative but is not counted again in the insertion/deletion totals. A zero precision/recall denominator is unavailable; F1 is unavailable only when its denominator is zero.

The summary and error details show raw and normalized references separately. With no successful/partial model run, model error counts and scores are unavailable, not fabricated. `SAVE_REPORT = True` saves every selected transcript using normalized scored labels, plus original STRING labels tagged SKIP. Each scored expected/predicted observation has an `error_type` tag (COR, DEL, INS, or SUB) and an index into paired comparisons. Predictions include recorded provenance: concept audit evidence/spans, reasons, model decisions, and linked request metadata; absent evidence is not invented. Insertions have no expected label. Missing/failed predictions have null scored tags, counts, and rates. The final JSON field, `micro_metrics`, pools TP/FP/FN across all selected transcripts, excludes SKIP labels, and includes missing/failed-row misses when any extraction is available. Every report uses a new `results\report_...\transcript_report.json` path; previous reports are never overwritten. `SAVE_RESULTS` separately enables the full audit export.

In [ ]:
metrics = evaluate(rows, predictions, registry)
if metrics['available']:
    summary = {}
    for view in ('raw', 'normalized'):
        score = metrics[view]['observation']
        summary[view] = {**metrics[view]['edit_counts'],
                         **{name: score[name] for name in ('precision', 'recall', 'f1')}}
    print('Observation-level errors and scores:')
    display(JSON(summary))
    for view in ('raw', 'normalized'):
        print(f'{view.capitalize()} observation comparison:')
        display(JSON(metrics[view]['alignment']))
else:
    print('UNAVAILABLE: no successful/partial model run; error counts and scores are not model results.')
display(JSON({'coverage': metrics['coverage'], 'counts': metrics['counts'],
              'source_candidate_coverage': metrics['diagnostics']['candidate_coverage'],
              'reference_issues': metrics['reference_issues'],
              'prediction_issues': metrics['prediction_issues'],
              'failures': metrics['failures']}))
if SAVE_RESULTS:
    if not predictions:
        raise RuntimeError('No model predictions to export; run JEV first.')
    run_dir = ROOT / 'results' / ('run_' + uuid4().hex[:12])
    save_run(run_dir, predictions, metrics, {
        'dataset_manifest': dataset.manifest,
        'split': SPLIT,
        'selected_row_ids': [row['id'] for row in rows],
        'requested_model': MODEL,
        'enabled_value_types': list(ENABLED_VALUE_TYPES),
        'normalization_policy': 'documented-unambiguous-reference-only-v1',
    })
    print('Saved run:', run_dir)
if SAVE_REPORT:
    source_rows_by_id = {row['id']: row for row in dataset.splits[SPLIT]}
    report_path = save_transcript_report(
        ROOT / 'results' / ('report_' + uuid4().hex[:12]),
        [{**source_rows_by_id[row['id']], 'split': SPLIT} for row in rows],
        predictions, registry,
        metadata={'requested_model': MODEL, 'dataset_manifest': dataset.manifest},
    )
    print('Saved transcript report:', report_path)